# Distribution-Aware OWOD Colab Experiment Notebook

This notebook is the single orchestration notebook for DAOWOD. It only clones repositories, installs dependencies, validates the protocol, and launches the experiment through the repository package.

In [ ]:
#@title Experiment parameters
SEED = 0  #@param {type:"integer"}
ROUNDS = 1  #@param {type:"integer"}
BUDGET = 10  #@param {type:"integer"}
VARIANTS = ['random', 'full']  #@param {type:"raw"}
COHERENCE_POWERS = [1.0]  #@param {type:"raw"}
OUTPUT_DIRECTORY = '/content/drive/MyDrive/DAOWOD/outputs'  #@param {type:"string"}
CHECKPOINT = '/content/drive/MyDrive/DAOWOD/checkpoints/MOWODB/t1.pth'  #@param {type:"string"}
DATASET = 'OWOD'  #@param {type:"string"}
DEVICE = 'cuda'  #@param {type:"string"}
REPO_URL = 'https://github.com/gubiczam/distribution-aware-owod.git'  #@param {type:"string"}
REPO_COMMIT = 'HEAD'  #@param {type:"string"}
PROB_REPO_URL = 'https://github.com/gubiczam/PROB.git'  #@param {type:"string"}
PROB_COMMIT = 'HEAD'  #@param {type:"string"}

import json
import shutil
import subprocess
import sys
from pathlib import Path
from types import SimpleNamespace

import pandas as pd
import yaml

def run_cmd(command, *, cwd=None, timeout=1800, check=True):
    command = [str(part) for part in command]
    print('$', ' '.join(command))
    result = subprocess.run(command, cwd=str(cwd) if cwd else None, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, timeout=timeout, check=False)
    if result.stdout:
        print(result.stdout[-8000:])
    if check and result.returncode != 0:
        raise subprocess.CalledProcessError(result.returncode, command, output=result.stdout)
    return result

ROOT = Path('/content')
DAOWOD_PATH = ROOT / 'distribution-aware-owod'
PROB_PATH = ROOT / 'PROB'
OUTPUT_ROOT = Path(OUTPUT_DIRECTORY)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print('Experiment root:', OUTPUT_ROOT)
print('Seed:', SEED)
print('Rounds:', ROUNDS)
print('Variants:', VARIANTS)
print('Coherence powers:', COHERENCE_POWERS)

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
print('Drive mounted.')

In [ ]:
if DAOWOD_PATH.exists():
    shutil.rmtree(DAOWOD_PATH)
if PROB_PATH.exists():
    shutil.rmtree(PROB_PATH)

run_cmd(['git', 'clone', REPO_URL, str(DAOWOD_PATH)], timeout=1200)
run_cmd(['git', '-C', str(DAOWOD_PATH), 'checkout', REPO_COMMIT], timeout=1200)
run_cmd(['git', 'clone', PROB_REPO_URL, str(PROB_PATH)], timeout=1200)
run_cmd(['git', '-C', str(PROB_PATH), 'checkout', PROB_COMMIT], timeout=1200)

print('Repository commit:', run_cmd(['git', '-C', str(DAOWOD_PATH), 'rev-parse', '--short', 'HEAD']).stdout.strip())
print('PROB commit:', run_cmd(['git', '-C', str(PROB_PATH), 'rev-parse', '--short', 'HEAD']).stdout.strip())

In [ ]:
run_cmd([sys.executable, '-m', 'pip', 'install', '--quiet', f'{DAOWOD_PATH}[dev]'], timeout=1800)
sys.path.insert(0, str(DAOWOD_PATH / 'src'))
import daowod
from daowod.config import load_config
print('DAOWOD import OK from', daowod.__file__)

In [ ]:
config_path = DAOWOD_PATH / 'configs' / 'experiment.yaml'
config = load_config(config_path)

base_output_dir = str(OUTPUT_ROOT / f'seed_{SEED}')
config_override = {
    'name': 'colab-experiment',
    'active_learning': {
        'rounds': ROUNDS,
        'strategy': VARIANTS[0],
        'budget': BUDGET,
        'initial_images': max(2, min(20, BUDGET)),
        'budget_per_round': BUDGET,
        'seeds': [SEED],
    },
    'acquisition': {
        'strategies': VARIANTS,
        'uncertainty_mode': 'ambiguity',
        'pseudo_label_source': 'cluster',
        'cluster_count': 20,
        'neighbour_count': 5,
        'top_k': 3,
        'weights': {
            'uncertainty': 0.3,
            'novelty': 0.2,
            'rarity': 0.5,
            'coherence_power': 1.0,
            'rarity_power': 1.0,
        },
    },
    'dataset': {
        'image_set_path': str(config.dataset.image_set_path),
        'annotations_dir': str(config.dataset.annotations_dir),
        'unknown_classes': list(config.dataset.unknown_classes),
        'long_tail': {
            'enabled': True,
            'imbalance_ratio': 50.0,
        },
    },
    'prob': {
        'repository_path': str(PROB_PATH),
        'initial_checkpoint': CHECKPOINT,
        'train_command': config.prob.train_command,
        'predict_command': config.prob.predict_command,
        'evaluate_command': config.prob.evaluate_command,
        'timeout_seconds': config.prob.timeout_seconds,
    },
    'output_dir': base_output_dir,
}

override_path = DAOWOD_PATH / 'configs' / 'colab_override.yaml'
override_path.write_text(yaml.safe_dump(config_override, sort_keys=False), encoding='utf-8')
config = load_config(override_path)

print('Configuration override prepared.')


In [ ]:
required_paths = [Path(CHECKPOINT), Path(config.dataset.image_set_path), Path(config.dataset.annotations_dir)]
missing = [str(path) for path in required_paths if not path.exists()]
if missing:
    raise FileNotFoundError('Missing required experiment assets: ' + ', '.join(missing))

if not Path(PROB_PATH).exists():
    raise FileNotFoundError('PROB repository is missing.')

print('Preflight validation passed.')

In [ ]:
from daowod.experiment import ActiveLearningExperiment
from daowod.prob_adapter import ProbAdapter

metrics_path = OUTPUT_ROOT / 'metrics.csv'
selections_path = OUTPUT_ROOT / 'selections.json'

if USE_RESUME and metrics_path.exists() and selections_path.exists():
    metrics_df = pd.read_csv(metrics_path)
    selections_df = pd.read_json(selections_path, orient='records')
    result = SimpleNamespace(
        metrics=metrics_df.to_dict(orient='records'),
        selections=selections_df.to_dict(orient='records'),
        output_dir=OUTPUT_ROOT,
    )
    print('Resumed from existing outputs:', OUTPUT_ROOT)
else:
    adapter = ProbAdapter(
        repository_path=PROB_PATH,
        train_command=config.prob.train_command,
        predict_command=config.prob.predict_command,
        evaluate_command=config.prob.evaluate_command,
        timeout_seconds=config.prob.timeout_seconds,
    )

    experiment = ActiveLearningExperiment(config, detector=adapter)
    result = experiment.run()

    print('Experiment completed. Metrics rows:', len(result.metrics))
    print('Selection rows:', len(result.selections))
    print('Output directory:', result.output_dir)


In [ ]:
summary_path = OUTPUT_ROOT / 'summary.csv'
metrics_path = OUTPUT_ROOT / 'metrics.csv'
selections_path = OUTPUT_ROOT / 'selections.json'

if result.metrics:
    pd.DataFrame(result.metrics).to_csv(metrics_path, index=False)
if result.selections:
    pd.DataFrame(result.selections).to_json(selections_path, orient='records', indent=2)

if result.metrics:
    pd.DataFrame(result.metrics).to_csv(summary_path, index=False)

print('Saved summary CSV:', summary_path)
print('Saved metrics CSV:', metrics_path)
print('Saved selections JSON:', selections_path)
